# 02. 리뷰 전처리

**목적:** 원본 리뷰를 정제하고 Komoran 명사 키워드와 안정적인 `review_id`를 생성합니다.

**입력:** `outputs/data/raw/acne_cream_reviews.csv`  
**출력:** `outputs/data/processed/Ntoken_review.csv`, 전처리 요약·키워드 그래프

기존 순서(정제 → 피부 정보 보완 → 형태소 분석 → 단어 합성 → 저장)를 유지합니다. 원본 CSV는 수정하지 않습니다. 아래 셀을 위에서부터 실행하세요. 패키지는 사전에 `requirements.txt`로 설치하며 Komoran에는 Java가 필요합니다.

## 1. 환경과 입출력 경로
난수와 파일 경로를 한곳에서 관리합니다. 그래프는 결과물 폴더에만 저장합니다.

In [1]:
from pathlib import Path
import sys

# Jupyter 실행 위치에 상관없이 프로젝트 내부에서 루트를 탐색합니다.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "code/common/project_utils.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더 내부에서 실행하세요.")
common_dir = str(PROJECT_ROOT / "code/common")
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)
from project_utils import SCHEMA_VERSION, add_review_ids, merge_sentiment, known_value, split_concerns

import re
import json
from collections import Counter
import pandas as pd
from konlpy.tag import Komoran
import matplotlib.pyplot as plt
from matplotlib import font_manager

INPUT_CSV = PROJECT_ROOT / "outputs/data/raw/acne_cream_reviews.csv"
OUTPUT_CSV = PROJECT_ROOT / "outputs/data/processed/Ntoken_review.csv"
FIGURE_DIR = PROJECT_ROOT / "outputs/figures/preprocessing"
REPORT_DIR = PROJECT_ROOT / "outputs/reports/preprocessing"
for folder in (OUTPUT_CSV.parent, FIGURE_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
# 실제 설치된 한글 폰트를 선택합니다. 폰트 이름을 파일 경로로 전달하지 않습니다.
font_names = {f.name for f in font_manager.fontManager.ttflist}
KOREAN_FONT = next((f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic"] if f in font_names), None)
if KOREAN_FONT:
    plt.rcParams["font.family"] = KOREAN_FONT
plt.rcParams["axes.unicode_minus"] = False


## 2. 기본 정제와 중복 제거

기존과 같이 한글 리뷰를 분석 대상으로 삼습니다. 숫자·영문 정보는 이 과정에서 제외되며 원본 CSV에는 보존됩니다. 사용자·제품·리뷰가 같은 중복은 한 건만 남깁니다. 유효하지 않은 평점은 추천 학습에 넣지 않습니다.

`review_id`는 제품명·작성자·정제 리뷰·날짜의 SHA-256입니다. 이후 파일의 행 순서가 바뀌어도 이 ID로 정확하게 병합합니다.

In [2]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def load_and_clean(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    required = {"product_name", "customer_name", "review", "date", "rating"}
    if not required.issubset(df.columns):
        raise ValueError(f"필수 컬럼 누락: {required - set(df.columns)}")
    original_count = len(df)
    for col in ["product_name", "customer_name", "skin_type", "skin_tone", "skin_concerns", "review", "gender"]:
        if col not in df:
            df[col] = ""
        df[col] = df[col].map(normalize_text)
    df["review"] = df.review.map(lambda s: re.sub(r"\s+", " ", re.sub(r"[^ ㄱ-ㅣ가-힣]+", " ", s)).strip())
    df["rating"] = pd.to_numeric(df.rating, errors="coerce")
    df["date"] = pd.to_datetime(df.date, errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    valid = df.review.ne("") & df.rating.between(1, 5) & df.product_name.ne("") & df.customer_name.ne("")
    removed_invalid = int((~valid).sum())
    df = df.loc[valid].copy()
    before = len(df)
    df = df.drop_duplicates(["product_name", "customer_name", "review"]).reset_index(drop=True)
    report = {"raw_rows": original_count, "invalid_rows_removed": removed_invalid,
              "duplicate_rows_removed": before - len(df), "processed_rows": len(df)}
    df = df.drop(columns=["product_brand", "product_link", "rating_text"], errors="ignore")
    return add_review_ids(df), report


df, preprocessing_report = load_and_clean(INPUT_CSV)
print(preprocessing_report)
display(df.head(3))


{'raw_rows': 12347, 'invalid_rows_removed': 30, 'duplicate_rows_removed': 522, 'processed_rows': 11795}


,product_name,customer_name,skin_type,skin_tone,skin_concerns,review,date,rating,gender,review_id
0,라로슈포제 시카플라스트 멀티 리페어 크림 100ml (+15ml+시카밤3ml+시카선...,vwaaang,건성,쿨톤,각질 / 모공,제품을 만난 이뢰로 계속 재구매중입니다 흡수가 빠르고 얼굴에 자고 일어나면 얼굴에 ...,2025-09-17,5.0,여성,5b74adc080b6f4bcc4e86c3990bad2e674c392b75ff500...
1,라로슈포제 시카플라스트 멀티 리페어 크림 100ml (+15ml+시카밤3ml+시카선...,나이거가사로써도돼,건성,웜톤,민감성 / 트러블,아마도 수부지인 피부인데 이거 바르고 피부 다 뒤집어져서 한번 쓰고 바로 다른 잘맞...,2025-09-07,1.0,여성,e8a6e3e2fe38be120fffe006e2c9e8fdebedf9cefc855d...
2,라로슈포제 시카플라스트 멀티 리페어 크림 100ml (+15ml+시카밤3ml+시카선...,찍지말라옹,복합성,겨울쿨톤,각질 / 모공,와 정말 순하네요 가격이 쎄서 살까말까하다가 결국 샀는데 후회 도 없어요 저는 원에...,2025-09-25,5.0,여성,0a2c0303c714ef64cbb31f7f3f5f3422266ced9d42c061...


## 3. 누락된 피부 타입 보완

명시된 피부 타입은 그대로 사용합니다. 비어 있는 경우에만 리뷰에서 최초로 발견된 피부 타입을 채웁니다. `수부지`는 기존 정책대로 `지성`으로 통합합니다. 이는 키워드 추정이므로 정확한 사용자 진단으로 해석하지 않습니다. 추정 여부는 `skin_type_inferred`에 남깁니다.

In [3]:
TYPE_PATTERN = re.compile(r"(트러블성|민감성|복합성|약건성|수부지|건성|지성|중성)")

def extract_skin_type(text):
    match = TYPE_PATTERN.search(str(text))
    if not match:
        return ""
    value = match.group(1)
    return "지성" if value == "수부지" else value

missing_skin = ~df.skin_type.map(known_value)
df["skin_type_inferred"] = False
inferred = df.loc[missing_skin, "review"].map(extract_skin_type)
df.loc[missing_skin, "skin_type"] = inferred
df.loc[missing_skin, "skin_type_inferred"] = inferred.ne("")
preprocessing_report["inferred_skin_rows"] = int(df.skin_type_inferred.sum())
preprocessing_report["missing_skin_rows"] = int((~df.skin_type.map(known_value)).sum())
print("피부 타입 보완:", preprocessing_report["inferred_skin_rows"])


피부 타입 보완: 1252


## 4. Komoran 형태소 분석과 명사 키워드

추천 콘텐츠에 사용할 명사(`NNG`, `NNP`)를 추출합니다. 감성 분석은 별도 노트북에서 리뷰 본문을 사용하므로 여기서 동사·형용사를 키워드에서 제외해도 감성 모델의 입력은 유지됩니다.

In [4]:
user_dict = PROJECT_ROOT / "resources/preprocessing/user_dict.txt"
komoran = Komoran(userdic=str(user_dict)) if user_dict.exists() else Komoran()
df["tokens"] = df.review.map(komoran.pos)
Ntoken_list = [[word for word, tag in tokens if tag in {"NNG", "NNP"}] for tokens in df.tokens]
print("형태소 분석 완료:", len(df))


형태소 분석 완료: 11795


## 5. 복합어 합성 및 불용어 제거

긴 패턴부터 적용해 `수분 크림`, `좁쌀 여드름` 등의 키워드를 합칩니다. 원래 토큰 리스트를 직접 변경하지 않아 셀 재실행 시 결과가 누적되지 않습니다.

In [5]:
COMPOUNDS = {
    ("아", "벤", "느"): "아벤느", ("리얼", "베리", "어"): "리얼베리어",
    ("수분", "부족", "지성"): "수부지", ("피부", "진정", "효과"): "진정",
    ("민", "감성", "피부"): "민감성", ("수분", "크림"): "수분크림",
    ("수분", "감"): "수분감", ("진정", "효과"): "진정효과",
    ("속", "건조"): "속건조", ("좁쌀", "여드름"): "좁쌀여드름",
}

def compound_words(tokens):
    result, i = [], 0
    while i < len(tokens):
        for n in (3, 2):
            candidate = tuple(tokens[i:i+n])
            if candidate in COMPOUNDS:
                result.append(COMPOUNDS[candidate])
                i += n
                break
        else:
            result.append(tokens[i])
            i += 1
    return result

stopwords = {"아벤느", "리얼베리어", "바이오힐보", "라로슈포", "올리브영"}
stopwords_path = PROJECT_ROOT / "resources/preprocessing/stopwords.txt"
if stopwords_path.exists():
    stopwords.update(w.strip() for w in stopwords_path.read_text(encoding="utf-8-sig").splitlines() if w.strip())
df["Ntoken_review"] = [[w for w in compound_words(tokens) if w not in stopwords and len(w) > 1]
                       for tokens in Ntoken_list]
display(df[["review_id", "review", "Ntoken_review"]].head(3))


,review_id,review,Ntoken_review
0,5b74adc080b6f4bcc4e86c3990bad2e674c392b75ff500...,제품을 만난 이뢰로 계속 재구매중입니다 흡수가 빠르고 얼굴에 자고 일어나면 얼굴에 ...,"[제품, 구매, 흡수, 얼굴, 얼굴, 분기, 여드름, 여드름]"
1,e8a6e3e2fe38be120fffe006e2c9e8fdebedf9cefc855d...,아마도 수부지인 피부인데 이거 바르고 피부 다 뒤집어져서 한번 쓰고 바로 다른 잘맞...,"[부지, 피부, 피부, 사람]"
2,0a2c0303c714ef64cbb31f7f3f5f3422266ced9d42c061...,와 정말 순하네요 가격이 쎄서 살까말까하다가 결국 샀는데 후회 도 없어요 저는 원에...,"[가격, 후회]"


## 6. 결과 저장과 키워드 확인

CSV와 요약을 먼저 저장한 후 그래프를 그립니다. 시각화 오류가 전처리 데이터 저장을 막지 않도록 순서를 분리했습니다. 워드클라우드는 한글 폰트가 있을 때만 생성합니다.

In [6]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
(REPORT_DIR / "summary.json").write_text(json.dumps(preprocessing_report, ensure_ascii=False, indent=2), encoding="utf-8")
counts = Counter(w for tokens in df.Ntoken_review for w in tokens)
if counts:
    top = pd.Series(dict(counts.most_common(20))).sort_values()
    fig, ax = plt.subplots(figsize=(9, 6))
    top.plot.barh(ax=ax)
    ax.set_title("상위 20개 리뷰 키워드")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "top20_keywords.png", dpi=160)
    plt.show()
    plt.close(fig)
    if KOREAN_FONT:
        from wordcloud import WordCloud
        font_path = font_manager.findfont(KOREAN_FONT)
        cloud = WordCloud(font_path=font_path, width=1000, height=500,
                          background_color="white", random_state=42).generate_from_frequencies(counts)
        cloud.to_file(str(FIGURE_DIR / "wordcloud.png"))
print("저장 완료:", OUTPUT_CSV)


/var/folders/ts/g7dt7hs11lvb1n__nmfls1hw0000gn/T/ipykernel_28285/554232882.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


저장 완료: /Users/Shared/Cosmetic_recommendation-main/outputs/data/processed/Ntoken_review.csv
